In [1]:
# Audit: setup

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from IPython.display import display




cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [cwd, *cwd.parents]
        if (path / "data" / "step3_multimodal").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the predicting-predictability project.\n"
        f"Current working directory: {cwd}\n\n"
        "Save this notebook somewhere inside the project folder."
    )


STEP3_DIR = (
    PROJECT_ROOT
    / "data"
    / "step3_multimodal"
)

STEP4_DIR = (
    STEP3_DIR
    / "step4_expanded_n20"
)

AUDIT_DIR = (
    STEP4_DIR
    / "predictability_direction_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)




ROW_ERRORS_PATH = (
    STEP3_DIR
    / "step3_bowtie_row_errors_latent256_final_with_source_normalized.csv"
)

SCORE_PATH = (
    STEP3_DIR
    / "step3_compound_predictability_latent256_final_scores.csv"
)

OOF_PATH = (
    STEP4_DIR
    / "step4_extra_trees_n20_oof_predictions.csv"
)

CLEAN_MOA_PATH = (
    STEP4_DIR
    / "step4_n20_full_clean_compound_moa.csv"
)


required_paths = {
    "Source-normalized row errors": ROW_ERRORS_PATH,
    "Final compound scores": SCORE_PATH,
    "Final out-of-fold predictions": OOF_PATH,
    "Clean compound–MoA table": CLEAN_MOA_PATH,
}


print("Project root:")
print(PROJECT_ROOT)

print("\nRequired audit files")
print("=" * 80)

missing_files = []

for description, path in required_paths.items():
    exists = path.exists()

    print(
        f"{description}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    print(path)
    print()

    if not exists:
        missing_files.append(
            (description, path)
        )


if missing_files:
    missing_text = "\n\n".join(
        f"{description}:\n{path}"
        for description, path in missing_files
    )

    raise FileNotFoundError(
        "The audit cannot continue because these files are missing:\n\n"
        f"{missing_text}"
    )

Project root:
/Users/mell/predicting-predictability

Required audit files
Source-normalized row errors: FOUND
/Users/mell/predicting-predictability/data/step3_multimodal/step3_bowtie_row_errors_latent256_final_with_source_normalized.csv

Final compound scores: FOUND
/Users/mell/predicting-predictability/data/step3_multimodal/step3_compound_predictability_latent256_final_scores.csv

Final out-of-fold predictions: FOUND
/Users/mell/predicting-predictability/data/step3_multimodal/step4_expanded_n20/step4_extra_trees_n20_oof_predictions.csv

Clean compound–MoA table: FOUND
/Users/mell/predicting-predictability/data/step3_multimodal/step4_expanded_n20/step4_n20_full_clean_compound_moa.csv



In [2]:
# Load all saved audit inputs

scores = pd.read_csv(
    SCORE_PATH,
    low_memory=False,
)

row_errors = pd.read_csv(
    ROW_ERRORS_PATH,
    low_memory=False,
)

oof_results = pd.read_csv(
    OOF_PATH,
    low_memory=False,
)

compound_moa = pd.read_csv(
    CLEAN_MOA_PATH,
    low_memory=False,
)


print("Loaded tables")
print("=" * 80)

print(
    f"Final Step 3 scores: "
    f"{scores.shape[0]:,} rows × {scores.shape[1]} columns"
)

print(
    f"Signature-level row errors: "
    f"{row_errors.shape[0]:,} rows × {row_errors.shape[1]} columns"
)

print(
    f"Step 4 out-of-fold results: "
    f"{oof_results.shape[0]:,} rows × {oof_results.shape[1]} columns"
)

print(
    f"Compound–MoA table: "
    f"{compound_moa.shape[0]:,} rows × {compound_moa.shape[1]} columns"
)


print("\nFinal score columns:")
print(scores.columns.tolist())

print("\nOOF result columns:")
print(oof_results.columns.tolist())

print("\nCompound–MoA columns:")
print(compound_moa.columns.tolist())

Loaded tables
Final Step 3 scores: 6,605 rows × 30 columns
Signature-level row errors: 112,974 rows × 31 columns
Step 4 out-of-fold results: 801 rows × 10 columns
Compound–MoA table: 1,002 rows × 8 columns

Final score columns:
['compound_embedding_row_index', 'compound_name', 'n_profiles', 'mean_total_error', 'variance_total_error', 'std_total_error', 'median_total_error', 'mean_signature_error', 'variance_signature_error', 'std_signature_error', 'mean_compound_error', 'variance_compound_error', 'mean_numeric_error', 'mean_categorical_error', 'mean_signature_reconstruction_loss', 'mean_compound_reconstruction_bce', 'mean_signature_sign_accuracy', 'mean_compound_bitwise_accuracy', 'error_percentile', 'predictability_score', 'mean_source_normalized_error_z', 'variance_source_normalized_error_z', 'std_source_normalized_error_z', 'median_source_normalized_error_z', 'n_sources', 'source_normalized_error_percentile', 'source_normalized_predictability_score', 'profile_count_bin', 'raw_error_

In [3]:
# Audit 1: Does the saved final score exactly match the intended inversion?

TARGET_COL = "source_normalized_inverted_score"
ERROR_COL = "mean_source_normalized_error_z"


required_score_columns = {
    "compound_embedding_row_index",
    "compound_name",
    "n_profiles",
    ERROR_COL,
    TARGET_COL,
}

missing_score_columns = (
    required_score_columns
    - set(scores.columns)
)

if missing_score_columns:
    raise KeyError(
        "The final score table is missing:\n"
        f"{sorted(missing_score_columns)}"
    )


audit_scores = scores.copy()

for column in [
    "n_profiles",
    ERROR_COL,
    TARGET_COL,
]:
    audit_scores[column] = pd.to_numeric(
        audit_scores[column],
        errors="coerce",
    )


audit_scores = audit_scores.dropna(
    subset=[
        ERROR_COL,
        TARGET_COL,
    ]
).copy()


# This reproduces the exact transformation from July Wraps

source_error = audit_scores[
    ERROR_COL
].copy()

q01 = source_error.quantile(0.01)
q99 = source_error.quantile(0.99)

source_error_clipped = source_error.clip(
    lower=q01,
    upper=q99,
)

error_range = (
    source_error_clipped.max()
    - source_error_clipped.min()
)

if error_range == 0:
    raise ValueError(
        "The clipped source-normalized error has zero range."
    )


audit_scores[
    "recomputed_source_normalized_inverted_score"
] = 1 - (
    (
        source_error_clipped
        - source_error_clipped.min()
    )
    / error_range
)


audit_scores["saved_minus_recomputed"] = (
    audit_scores[TARGET_COL]
    - audit_scores[
        "recomputed_source_normalized_inverted_score"
    ]
)


max_score_difference = (
    audit_scores[
        "saved_minus_recomputed"
    ]
    .abs()
    .max()
)

mean_score_difference = (
    audit_scores[
        "saved_minus_recomputed"
    ]
    .abs()
    .mean()
)


rho_error_score = spearmanr(
    audit_scores[ERROR_COL],
    audit_scores[TARGET_COL],
).correlation


print("FINAL SCORE INVERSION AUDIT")
print("=" * 80)

print(
    f"1st-percentile error cutoff: {q01:.6f}"
)

print(
    f"99th-percentile error cutoff: {q99:.6f}"
)

print(
    f"Maximum absolute difference between saved and "
    f"recomputed scores: {max_score_difference:.12g}"
)

print(
    f"Mean absolute difference: "
    f"{mean_score_difference:.12g}"
)

print(
    f"Spearman correlation between source-normalized error "
    f"and final score: {rho_error_score:.6f}"
)


if max_score_difference < 1e-8:
    print(
        "\nPASS: The saved target matches the intended "
        "1 − min-max-scaled-error formula."
    )
else:
    print(
        "\nFAIL: The saved target does not exactly match "
        "the intended inversion formula."
    )


if rho_error_score < 0:
    print(
        "PASS: Larger reconstruction error corresponds "
        "to a lower final predictability score."
    )
else:
    print(
        "WARNING: Larger error does not correspond to "
        "a lower score. The direction may be flipped."
    )


display(
    audit_scores[
        [
            "compound_name",
            "n_profiles",
            ERROR_COL,
            TARGET_COL,
            "recomputed_source_normalized_inverted_score",
            "saved_minus_recomputed",
        ]
    ]
    .sort_values(
        ERROR_COL,
        ascending=True,
    )
    .head(10)
)


display(
    audit_scores[
        [
            "compound_name",
            "n_profiles",
            ERROR_COL,
            TARGET_COL,
            "recomputed_source_normalized_inverted_score",
            "saved_minus_recomputed",
        ]
    ]
    .sort_values(
        ERROR_COL,
        ascending=False,
    )
    .head(10)
)

FINAL SCORE INVERSION AUDIT
1st-percentile error cutoff: -0.871473
99th-percentile error cutoff: 4.087702
Maximum absolute difference between saved and recomputed scores: 1.11022302463e-16
Mean absolute difference: 1.02783679141e-17
Spearman correlation between source-normalized error and final score: -0.999999

PASS: The saved target matches the intended 1 − min-max-scaled-error formula.
PASS: Larger reconstruction error corresponds to a lower final predictability score.


,compound_name,n_profiles,mean_source_normalized_error_z,source_normalized_inverted_score,recomputed_source_normalized_inverted_score,saved_minus_recomputed
0,"(1R,9S,12S,13R,14S,17R,18E,21S,23S,24R,25S,27R...",2,-1.281762,1.0,1.0,0.0
1,"Emate; [(8R,9S,13S,14S)-13-methyl-17-oxo-7,8,9...",2,-1.243400,1.0,1.0,0.0
2,"5-[(5-chloro-1H-pyrrolo[2,3-b]pyridin-3-yl)met...",1,-1.241747,1.0,1.0,0.0
3,"Halobetasol Propionate; [(6S,8S,9R,10S,11S,13S...",4,-1.227075,1.0,1.0,0.0
6,"(3S,6S,9S,12R,15S,18S,21S,24S,30S,33S)-15,30-d...",4,-1.200180,1.0,1.0,0.0
12,4-Aminobenzamide; 4-aminobenzamide,4,-1.182204,1.0,1.0,0.0
14,"(2S,3S)-2-(2,4-difluorophenyl)-3-(5-fluoropyri...",4,-1.175150,1.0,1.0,0.0
260,isoflupredone,3,-1.162108,1.0,1.0,0.0
21,"(2S,3S)-2-amino-3-methylpentanoic acid; L-Isol...",3,-1.154702,1.0,1.0,0.0
50,(E)-3-(4-((Z)-1-(4-Hydroxyphenyl)-2-phenylbut-...,1,-1.129936,1.0,1.0,0.0


,compound_name,n_profiles,mean_source_normalized_error_z,source_normalized_inverted_score,recomputed_source_normalized_inverted_score,saved_minus_recomputed
6588,"(3R,4R)-N-[3-(dimethylamino)propyl]-3-[4-(hydr...",1,6.429506,0.0,0.0,0.0
6582,"[(1R,2R,3S,6S,7S,9S,10R,11R,12R,13S,14R)-2,6,9...",1,6.147671,0.0,0.0,0.0
6604,ethyl N-[4-[(E)-3-(4-morpholin-4-ylquinolin-2-...,2,6.015509,0.0,0.0,0.0
6578,"N-[3-chloro-4-(1,3-dioxo-2-azaspiro[4.5]decan-...",1,6.004314,0.0,0.0,0.0
6576,"(1R,9S,10S,11S)-10-(hydroxymethyl)-N,N-dimethy...",1,5.835187,0.0,0.0,0.0
6575,(1H-Indol-3-ylsulfanyl)-acetic acid; BRD-K7460...,1,5.686368,0.0,0.0,0.0
6574,MDL-72832; Mdl 72832,1,5.682112,0.0,0.0,0.0
6568,PF 04457845; PF-04457845,4,5.474434,0.0,0.0,0.0
6563,"2-[(4Z)-4-(furan-2-ylmethylidene)-2,5-dioxoimi...",1,5.395308,0.0,0.0,0.0
6561,NGB-2904; Ngb 2904,1,5.352614,0.0,0.0,0.0


In [4]:
# Audit 2: Recreate the compound-level source-normalized error from the signature-level row table

required_row_columns = {
    "compound_embedding_row_index",
    "compound_name",
    "signature_row_index",
    "source_normalized_total_error_z",
}

missing_row_columns = (
    required_row_columns
    - set(row_errors.columns)
)

if missing_row_columns:
    raise KeyError(
        "The source-normalized row-error table is missing:\n"
        f"{sorted(missing_row_columns)}"
    )


row_errors_audit = row_errors.copy()

row_errors_audit[
    "compound_embedding_row_index"
] = pd.to_numeric(
    row_errors_audit[
        "compound_embedding_row_index"
    ],
    errors="coerce",
)

row_errors_audit[
    "source_normalized_total_error_z"
] = pd.to_numeric(
    row_errors_audit[
        "source_normalized_total_error_z"
    ],
    errors="coerce",
)


row_errors_audit = row_errors_audit.dropna(
    subset=[
        "compound_embedding_row_index",
        "source_normalized_total_error_z",
    ]
).copy()


recomputed_compound_errors = (
    row_errors_audit
    .groupby(
        [
            "compound_embedding_row_index",
            "compound_name",
        ],
        dropna=False,
    )
    .agg(
        recomputed_n_profiles=(
            "signature_row_index",
            "count",
        ),
        recomputed_mean_source_normalized_error_z=(
            "source_normalized_total_error_z",
            "mean",
        ),
    )
    .reset_index()
)


aggregation_audit = (
    scores[
        [
            "compound_embedding_row_index",
            "compound_name",
            "n_profiles",
            ERROR_COL,
            TARGET_COL,
        ]
    ]
    .merge(
        recomputed_compound_errors,
        on=[
            "compound_embedding_row_index",
            "compound_name",
        ],
        how="inner",
    )
)


aggregation_audit["profile_count_difference"] = (
    aggregation_audit["n_profiles"]
    - aggregation_audit[
        "recomputed_n_profiles"
    ]
)

aggregation_audit["mean_error_difference"] = (
    aggregation_audit[ERROR_COL]
    - aggregation_audit[
        "recomputed_mean_source_normalized_error_z"
    ]
)


max_profile_difference = (
    aggregation_audit[
        "profile_count_difference"
    ]
    .abs()
    .max()
)

max_mean_error_difference = (
    aggregation_audit[
        "mean_error_difference"
    ]
    .abs()
    .max()
)


print("SIGNATURE-TO-COMPOUND AGGREGATION AUDIT")
print("=" * 80)

print(
    f"Compounds compared: "
    f"{len(aggregation_audit):,}"
)

print(
    f"Maximum profile-count difference: "
    f"{max_profile_difference}"
)

print(
    f"Maximum compound mean-error difference: "
    f"{max_mean_error_difference:.12g}"
)


if (
    max_profile_difference == 0
    and max_mean_error_difference < 1e-8
):
    print(
        "\nPASS: Signature-level source-normalized errors "
        "were aggregated into compound means correctly."
    )
else:
    print(
        "\nWARNING: The saved compound table does not "
        "perfectly match the row-level aggregation."
    )


display(
    aggregation_audit[
        [
            "compound_name",
            "n_profiles",
            "recomputed_n_profiles",
            ERROR_COL,
            "recomputed_mean_source_normalized_error_z",
            "mean_error_difference",
            TARGET_COL,
        ]
    ]
    .sort_values(
        "mean_error_difference",
        key=lambda values: values.abs(),
        ascending=False,
    )
    .head(20)
)

SIGNATURE-TO-COMPOUND AGGREGATION AUDIT
Compounds compared: 6,605
Maximum profile-count difference: 0
Maximum compound mean-error difference: 8.881784197e-16

PASS: Signature-level source-normalized errors were aggregated into compound means correctly.


,compound_name,n_profiles,recomputed_n_profiles,mean_source_normalized_error_z,recomputed_mean_source_normalized_error_z,mean_error_difference,source_normalized_inverted_score
6389,Enrofloxacin; enrofloxacin,5,5,3.612588,3.612588,-8.881784e-16,0.095805
5713,"2-[cyclohexyl-[2-[5-(3,4-dimethoxyphenyl)-2-te...",2,2,2.232273,2.232273,-8.881784e-16,0.374141
6240,"BRD-A98248982; N-[2-(3,4-dimethoxyphenyl)ethyl...",3,3,3.124542,3.124542,8.881784e-16,0.194218
6038,3-Amino-4-(4-chlorophenyl)-6-cyclopropyl-2-thi...,3,3,2.692783,2.692783,-8.881784e-16,0.281280
6598,(E)-3-(1H-benzo[d]imidazol-2-yl)-1-(6-chloro-2...,3,3,4.329535,4.329535,-8.881784e-16,0.000000
5235,Immethridine; immethridine,2,2,1.794960,1.794960,-6.661338e-16,0.462323
5861,"2-[[5-(2-Furanyl)-4-methyl-1,2,4-triazol-3-yl]...",5,5,2.402536,2.402536,4.440892e-16,0.339808
5042,"2,2-Dimethyl-N-(2,4,6-trimethoxyphenyl)dodecan...",10,10,1.643645,1.643645,-4.440892e-16,0.492835
5772,"(1,2-Dimethyl-3-imidazo[1,2-a]pyridin-4-iumyl)...",2,2,2.306416,2.306416,4.440892e-16,0.359190
5662,AZ-10417808,2,2,2.177007,2.177007,4.440892e-16,0.385285


In [6]:
# Audit 3: Did Step 4's "actual_predictability" exactly equal the final Step 3 score?

required_oof_columns = {
    "compound_embedding_row_index",
    "compound_name",
    "n_profiles",
    "actual_predictability",
    "predicted_predictability",
}

missing_oof_columns = (
    required_oof_columns
    - set(oof_results.columns)
)

if missing_oof_columns:
    raise KeyError(
        "The OOF result table is missing required columns:\n"
        f"{sorted(missing_oof_columns)}\n\n"
        f"Available columns:\n{oof_results.columns.tolist()}"
    )


required_step3_columns = {
    "compound_embedding_row_index",
    TARGET_COL,
    ERROR_COL,
}

missing_step3_columns = (
    required_step3_columns
    - set(scores.columns)
)

if missing_step3_columns:
    raise KeyError(
        "The Step 3 score table is missing required columns:\n"
        f"{sorted(missing_step3_columns)}\n\n"
        f"Available columns:\n{scores.columns.tolist()}"
    )



oof_handoff_data = oof_results[
    [
        "compound_embedding_row_index",
        "compound_name",
        "n_profiles",
        "actual_predictability",
        "predicted_predictability",
    ]
].copy()


step3_handoff_data = scores[
    [
        "compound_embedding_row_index",
        TARGET_COL,
        ERROR_COL,
    ]
].copy()


for dataframe in [
    oof_handoff_data,
    step3_handoff_data,
]:
    dataframe[
        "compound_embedding_row_index"
    ] = pd.to_numeric(
        dataframe[
            "compound_embedding_row_index"
        ],
        errors="coerce",
    )


for column in [
    "n_profiles",
    "actual_predictability",
    "predicted_predictability",
]:
    oof_handoff_data[column] = pd.to_numeric(
        oof_handoff_data[column],
        errors="coerce",
    )


for column in [
    TARGET_COL,
    ERROR_COL,
]:
    step3_handoff_data[column] = pd.to_numeric(
        step3_handoff_data[column],
        errors="coerce",
    )


oof_handoff_data = (
    oof_handoff_data
    .dropna(
        subset=[
            "compound_embedding_row_index",
            "actual_predictability",
        ]
    )
    .drop_duplicates(
        subset="compound_embedding_row_index",
        keep="first",
    )
    .copy()
)


step3_handoff_data = (
    step3_handoff_data
    .dropna(
        subset=[
            "compound_embedding_row_index",
            TARGET_COL,
        ]
    )
    .drop_duplicates(
        subset="compound_embedding_row_index",
        keep="first",
    )
    .copy()
)


oof_handoff_data[
    "compound_embedding_row_index"
] = oof_handoff_data[
    "compound_embedding_row_index"
].astype(int)


step3_handoff_data[
    "compound_embedding_row_index"
] = step3_handoff_data[
    "compound_embedding_row_index"
].astype(int)


step4_handoff = (
    oof_handoff_data
    .merge(
        step3_handoff_data,
        on="compound_embedding_row_index",
        how="inner",
        validate="one_to_one",
    )
)


if step4_handoff.empty:
    raise ValueError(
        "No compounds matched between the Step 3 scores "
        "and Step 4 OOF results."
    )


step4_handoff[
    "target_handoff_difference"
] = (
    step4_handoff[
        "actual_predictability"
    ]
    - step4_handoff[TARGET_COL]
)


step4_handoff[
    "absolute_target_handoff_difference"
] = (
    step4_handoff[
        "target_handoff_difference"
    ]
    .abs()
)


max_handoff_difference = (
    step4_handoff[
        "absolute_target_handoff_difference"
    ]
    .max()
)


mean_handoff_difference = (
    step4_handoff[
        "absolute_target_handoff_difference"
    ]
    .mean()
)


median_handoff_difference = (
    step4_handoff[
        "absolute_target_handoff_difference"
    ]
    .median()
)


rho_handoff = spearmanr(
    step4_handoff[
        "actual_predictability"
    ],
    step4_handoff[TARGET_COL],
).correlation


pearson_handoff = (
    step4_handoff[
        [
            "actual_predictability",
            TARGET_COL,
        ]
    ]
    .corr(method="pearson")
    .iloc[0, 1]
)



HANDOFF_ATOL = 1e-6
HANDOFF_RTOL = 1e-7


handoff_passed = np.allclose(
    step4_handoff[
        "actual_predictability"
    ].to_numpy(),
    step4_handoff[
        TARGET_COL
    ].to_numpy(),
    atol=HANDOFF_ATOL,
    rtol=HANDOFF_RTOL,
    equal_nan=False,
)


inverse_handoff_match = np.allclose(
    step4_handoff[
        "actual_predictability"
    ].to_numpy(),
    (
        1
        - step4_handoff[
            TARGET_COL
        ].to_numpy()
    ),
    atol=HANDOFF_ATOL,
    rtol=HANDOFF_RTOL,
    equal_nan=False,
)



print("STEP 3 → STEP 4 TARGET HANDOFF AUDIT")
print("=" * 80)

print(
    f"Step 4 compounds: "
    f"{len(oof_handoff_data):,}"
)

print(
    f"Step 3 compounds available: "
    f"{len(step3_handoff_data):,}"
)

print(
    f"Compounds compared: "
    f"{len(step4_handoff):,}"
)

print(
    f"\nMaximum absolute target difference: "
    f"{max_handoff_difference:.12g}"
)

print(
    f"Mean absolute target difference: "
    f"{mean_handoff_difference:.12g}"
)

print(
    f"Median absolute target difference: "
    f"{median_handoff_difference:.12g}"
)

print(
    f"\nSpearman correlation: "
    f"{rho_handoff:.12f}"
)

print(
    f"Pearson correlation: "
    f"{pearson_handoff:.12f}"
)

print(
    f"\nAbsolute tolerance: "
    f"{HANDOFF_ATOL}"
)

print(
    f"Relative tolerance: "
    f"{HANDOFF_RTOL}"
)


if handoff_passed:

    print(
        "\nPASS: Step 4 used the final Step 3 score "
        "without any meaningful flipping or alteration."
    )

    print(
        "The tiny differences are consistent with "
        "floating-point or CSV rounding."
    )

elif inverse_handoff_match:

    print(
        "\nFAIL: Step 4 appears to have used the inverse "
        "of the final Step 3 score."
    )

else:

    print(
        "\nFAIL: Step 4's actual_predictability meaningfully "
        "differs from the final Step 3 score."
    )



print(
    "\nCompounds with the largest numerical differences:"
)

display(
    step4_handoff[
        [
            "compound_embedding_row_index",
            "compound_name",
            "n_profiles",
            ERROR_COL,
            TARGET_COL,
            "actual_predictability",
            "predicted_predictability",
            "target_handoff_difference",
            "absolute_target_handoff_difference",
        ]
    ]
    .sort_values(
        "absolute_target_handoff_difference",
        ascending=False,
    )
    .head(20)
)



print(
    "\nAbsolute target-difference summary:"
)

display(
    step4_handoff[
        "absolute_target_handoff_difference"
    ]
    .describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99,
        ]
    )
    .to_frame(
        name="absolute_difference"
    )
)

STEP 3 → STEP 4 TARGET HANDOFF AUDIT
Step 4 compounds: 801
Step 3 compounds available: 6,605
Compounds compared: 801

Maximum absolute target difference: 5.54209446113e-08
Mean absolute target difference: 1.80005992299e-08
Median absolute target difference: 1.65553546427e-08

Spearman correlation: 1.000000000000
Pearson correlation: 1.000000000000

Absolute tolerance: 1e-06
Relative tolerance: 1e-07

PASS: Step 4 used the final Step 3 score without any meaningful flipping or alteration.
The tiny differences are consistent with floating-point or CSV rounding.

Compounds with the largest numerical differences:


,compound_embedding_row_index,compound_name,n_profiles,mean_source_normalized_error_z,source_normalized_inverted_score,actual_predictability,predicted_predictability,target_handoff_difference,absolute_target_handoff_difference
72,5194,LY-2584702 (tosylate salt),144,-0.084945,0.841399,0.841399,0.807446,5.542094e-08,5.542094e-08
544,3444,Buparlisib; buparlisib,63,-0.290864,0.882922,0.882922,0.848925,5.484093e-08,5.484093e-08
420,2708,Ceritinib; ceritinib,50,-0.250534,0.874790,0.874790,0.814501,5.322735e-08,5.322735e-08
671,4161,"2-((3-(2,3-Dichlorophenoxy)propyl)amino)ethano...",23,0.596470,0.703995,0.703995,0.767383,5.317638e-08,5.317638e-08
27,5212,Lenalidomide (hemihydrate); Lenalidomide hemih...,143,-0.269661,0.878647,0.878647,0.844571,5.305143e-08,5.305143e-08
728,923,BMS-265246; Sdccgsbi-0654469.P001,46,-0.051208,0.834597,0.834597,0.819108,5.299581e-08,5.299581e-08
457,586,"Fluvastatin, (3R,5S)-; fluvastatin",47,-0.156718,0.855872,0.855872,0.820662,5.260905e-08,5.260905e-08
64,2711,(R)-9-(4-(1-aminopropan-2-yl)phenyl)-8-hydroxy...,143,-0.130241,0.850533,0.850533,0.818394,-5.260528e-08,5.260528e-08
674,5455,tioguanine,32,0.603197,0.702638,0.702638,0.834103,5.252430e-08,5.252430e-08
56,5186,Bosentan (hydrate); Bosentan Monohydrate,147,-0.346733,0.894188,0.894188,0.815768,-5.239469e-08,5.239469e-08



Absolute target-difference summary:


,absolute_difference
count,8.010000e+02
mean,1.800060e-08
std,1.255031e-08
min,0.000000e+00
50%,1.655535e-08
90%,3.488663e-08
95%,4.111398e-08
99%,5.252430e-08
max,5.542094e-08


In [7]:
# Audit 4: Rebuild MoA-level values and inspect HDAC inhibitors

moa_replacements = {
    "NFKB pathway inhibitor":
        "NFkB pathway inhibitor",

    "NF-kB pathway inhibitor":
        "NFkB pathway inhibitor",

    "Nfkb pathway inhibitor":
        "NFkB pathway inhibitor",

    "Tubulin inhibitor":
        "Tubulin polymerization inhibitor",

    "MTOR inhibitor":
        "mTOR inhibitor",

    "Mtor inhibitor":
        "mTOR inhibitor",

    "Topoisomerase inhibitor; topoisomerase inhibitor":
        "Topoisomerase inhibitor",

    "Glucocorticoid receptor agonist; glucocorticoid receptor agonist":
        "Glucocorticoid receptor agonist",
}


required_moa_columns = {
    "compound_embedding_row_index",
    "compound_name",
    "n_profiles",
    "clean_moa",
    "actual_predictability",
    "predicted_predictability",
}

missing_moa_columns = (
    required_moa_columns
    - set(compound_moa.columns)
)

if missing_moa_columns:
    raise KeyError(
        "The clean compound–MoA table is missing:\n"
        f"{sorted(missing_moa_columns)}"
    )


moa_audit = compound_moa.copy()

moa_audit["final_moa"] = (
    moa_audit["clean_moa"]
    .replace(moa_replacements)
    .astype("string")
    .str.strip()
)


for column in [
    "compound_embedding_row_index",
    "n_profiles",
    "actual_predictability",
    "predicted_predictability",
]:
    moa_audit[column] = pd.to_numeric(
        moa_audit[column],
        errors="coerce",
    )


moa_audit = (
    moa_audit
    .dropna(
        subset=[
            "compound_embedding_row_index",
            "final_moa",
            "actual_predictability",
            "predicted_predictability",
        ]
    )
    .loc[
        lambda dataframe:
        ~dataframe["final_moa"].isin(
            [
                "",
                "nan",
                "None",
                "unknown",
                "unclear",
            ]
        )
    ]
    .drop_duplicates(
        subset=[
            "compound_embedding_row_index",
            "final_moa",
        ]
    )
    .reset_index(drop=True)
)


MIN_COMPOUNDS = 5

moa_summary = (
    moa_audit
    .groupby(
        "final_moa",
        as_index=False,
    )
    .agg(
        n_compounds=(
            "compound_embedding_row_index",
            "nunique",
        ),
        total_profiles=(
            "n_profiles",
            "sum",
        ),
        median_profiles_per_compound=(
            "n_profiles",
            "median",
        ),
        observed_mean=(
            "actual_predictability",
            "mean",
        ),
        predicted_mean=(
            "predicted_predictability",
            "mean",
        ),
        observed_median=(
            "actual_predictability",
            "median",
        ),
        predicted_median=(
            "predicted_predictability",
            "median",
        ),
    )
)


moa_summary = (
    moa_summary[
        moa_summary["n_compounds"]
        >= MIN_COMPOUNDS
    ]
    .copy()
)


moa_summary["prediction_error"] = (
    moa_summary["predicted_mean"]
    - moa_summary["observed_mean"]
)

moa_summary["absolute_error"] = (
    moa_summary["prediction_error"]
    .abs()
)


moa_summary["observed_predictability_rank"] = (
    moa_summary["observed_mean"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


moa_summary["prediction_error_rank"] = (
    moa_summary["absolute_error"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


moa_summary = moa_summary.sort_values(
    "observed_mean",
    ascending=False,
).reset_index(drop=True)


hdac_summary = moa_summary[
    moa_summary["final_moa"]
    .str.contains(
        "HDAC",
        case=False,
        na=False,
    )
].copy()


print("MOA AUDIT")
print("=" * 80)

print(
    f"MoAs with at least {MIN_COMPOUNDS} compounds: "
    f"{len(moa_summary)}"
)

print("\nHDAC inhibitor result:")

display(
    hdac_summary[
        [
            "final_moa",
            "n_compounds",
            "total_profiles",
            "median_profiles_per_compound",
            "observed_mean",
            "predicted_mean",
            "prediction_error",
            "absolute_error",
            "observed_predictability_rank",
            "prediction_error_rank",
        ]
    ]
    .round(4)
)


print("\nMoAs with the highest actual multimodal proxy scores:")

display(
    moa_summary[
        [
            "final_moa",
            "n_compounds",
            "observed_mean",
            "predicted_mean",
            "absolute_error",
            "observed_predictability_rank",
        ]
    ]
    .head(15)
    .round(4)
)


print("\nMoAs with the lowest actual multimodal proxy scores:")

display(
    moa_summary[
        [
            "final_moa",
            "n_compounds",
            "observed_mean",
            "predicted_mean",
            "absolute_error",
            "observed_predictability_rank",
        ]
    ]
    .tail(15)
    .sort_values(
        "observed_mean",
        ascending=True,
    )
    .round(4)
)


print("\nMoAs predicted least accurately by the structure model:")

display(
    moa_summary[
        [
            "final_moa",
            "n_compounds",
            "observed_mean",
            "predicted_mean",
            "prediction_error",
            "absolute_error",
            "prediction_error_rank",
        ]
    ]
    .sort_values(
        "absolute_error",
        ascending=False,
    )
    .head(15)
    .round(4)
)

MOA AUDIT
MoAs with at least 5 compounds: 52

HDAC inhibitor result:


,final_moa,n_compounds,total_profiles,median_profiles_per_compound,observed_mean,predicted_mean,prediction_error,absolute_error,observed_predictability_rank,prediction_error_rank
7,HDAC inhibitor,19,3528,97.0,0.88,0.8184,-0.0616,0.0616,8,3



MoAs with the highest actual multimodal proxy scores:


,final_moa,n_compounds,observed_mean,predicted_mean,absolute_error,observed_predictability_rank
0,Proteasome inhibitor,7,0.9168,0.8353,0.0815,1
1,Topoisomerase inhibitor,22,0.9133,0.8957,0.0176,2
2,mTOR inhibitor,17,0.9124,0.8509,0.0615,3
3,Protein synthesis inhibitor,10,0.9021,0.8525,0.0496,4
4,Ribonucleotide reductase inhibitor,5,0.8917,0.8321,0.0597,5
5,PLK inhibitor,7,0.8904,0.8208,0.0695,6
6,ATPase inhibitor,10,0.8870,0.8660,0.0211,7
7,HDAC inhibitor,19,0.8800,0.8184,0.0616,8
8,Tubulin polymerization inhibitor,16,0.8748,0.8399,0.0349,9
9,PI3K inhibitor,19,0.8746,0.8368,0.0378,10



MoAs with the lowest actual multimodal proxy scores:


,final_moa,n_compounds,observed_mean,predicted_mean,absolute_error,observed_predictability_rank
51,GSK-3 inhibitor,5,0.7506,0.8054,0.0548,52
50,Retinoid receptor agonist,9,0.7606,0.8019,0.0413,51
49,Tyrosine kinase inhibitor,10,0.7706,0.8101,0.0394,50
48,Bcr-Abl kinase inhibitor,6,0.7714,0.8038,0.0324,49
47,Adrenergic receptor antagonist,6,0.7835,0.8004,0.0169,48
46,Adrenergic receptor agonist,7,0.7852,0.7910,0.0058,47
45,RAF inhibitor,5,0.7859,0.8116,0.0257,46
44,FGFR inhibitor,6,0.7872,0.8178,0.0306,45
43,PDGFR inhibitor,8,0.7884,0.8118,0.0234,44
42,PDGFR tyrosine kinase receptor inhibitor,9,0.7916,0.8178,0.0262,43



MoAs predicted least accurately by the structure model:


,final_moa,n_compounds,observed_mean,predicted_mean,prediction_error,absolute_error,prediction_error_rank
0,Proteasome inhibitor,7,0.9168,0.8353,-0.0815,0.0815,1
5,PLK inhibitor,7,0.8904,0.8208,-0.0695,0.0695,2
7,HDAC inhibitor,19,0.8800,0.8184,-0.0616,0.0616,3
2,mTOR inhibitor,17,0.9124,0.8509,-0.0615,0.0615,4
4,Ribonucleotide reductase inhibitor,5,0.8917,0.8321,-0.0597,0.0597,5
51,GSK-3 inhibitor,5,0.7506,0.8054,0.0548,0.0548,6
3,Protein synthesis inhibitor,10,0.9021,0.8525,-0.0496,0.0496,7
11,MEK inhibitor,10,0.8701,0.8270,-0.0431,0.0431,8
50,Retinoid receptor agonist,9,0.7606,0.8019,0.0413,0.0413,9
15,NFkB pathway inhibitor,10,0.8560,0.8161,-0.0399,0.0399,10


In [11]:
# Audit 5: Inspect the actual compounds making up the HDAC inhibitor average



score_columns_needed = [
    "compound_embedding_row_index",
    "mean_total_error",
    "mean_source_normalized_error_z",
    "source_normalized_inverted_score",
]

missing_score_columns = (
    set(score_columns_needed)
    - set(scores.columns)
)

if missing_score_columns:
    raise KeyError(
        "The Step 3 score table is missing required columns:\n"
        f"{sorted(missing_score_columns)}"
    )


scores_unique = (
    scores[
        score_columns_needed
    ]
    .copy()
)


scores_unique[
    "compound_embedding_row_index"
] = pd.to_numeric(
    scores_unique[
        "compound_embedding_row_index"
    ],
    errors="coerce",
)


for column in [
    "mean_total_error",
    "mean_source_normalized_error_z",
    "source_normalized_inverted_score",
]:
    scores_unique[column] = pd.to_numeric(
        scores_unique[column],
        errors="coerce",
    )


scores_unique = (
    scores_unique
    .dropna(
        subset=[
            "compound_embedding_row_index",
        ]
    )
    .drop_duplicates(
        subset="compound_embedding_row_index",
        keep="first",
    )
    .copy()
)


scores_unique[
    "compound_embedding_row_index"
] = scores_unique[
    "compound_embedding_row_index"
].astype(int)




hdac_compounds_base = (
    moa_audit[
        moa_audit["final_moa"]
        .str.contains(
            "HDAC",
            case=False,
            na=False,
        )
    ]
    [
        [
            "compound_embedding_row_index",
            "compound_name",
            "n_profiles",
            "final_moa",
            "actual_predictability",
            "predicted_predictability",
        ]
    ]
    .copy()
)


hdac_compounds_base[
    "compound_embedding_row_index"
] = pd.to_numeric(
    hdac_compounds_base[
        "compound_embedding_row_index"
    ],
    errors="coerce",
)


for column in [
    "n_profiles",
    "actual_predictability",
    "predicted_predictability",
]:
    hdac_compounds_base[column] = pd.to_numeric(
        hdac_compounds_base[column],
        errors="coerce",
    )


hdac_compounds_base = (
    hdac_compounds_base
    .dropna(
        subset=[
            "compound_embedding_row_index",
            "actual_predictability",
            "predicted_predictability",
        ]
    )
    .drop_duplicates(
        subset="compound_embedding_row_index",
        keep="first",
    )
    .copy()
)


hdac_compounds_base[
    "compound_embedding_row_index"
] = hdac_compounds_base[
    "compound_embedding_row_index"
].astype(int)




hdac_compounds = (
    hdac_compounds_base
    .merge(
        scores_unique,
        on="compound_embedding_row_index",
        how="left",
        validate="one_to_one",
    )
)




hdac_compounds["prediction_error"] = (
    hdac_compounds[
        "predicted_predictability"
    ]
    - hdac_compounds[
        "actual_predictability"
    ]
)


hdac_compounds["absolute_error"] = (
    hdac_compounds[
        "prediction_error"
    ]
    .abs()
)


hdac_compounds = (
    hdac_compounds
    .sort_values(
        "actual_predictability",
        ascending=False,
    )
    .reset_index(drop=True)
)



n_rows = len(
    hdac_compounds
)

n_unique_compounds = (
    hdac_compounds[
        "compound_embedding_row_index"
    ]
    .nunique()
)


if n_rows != n_unique_compounds:
    raise ValueError(
        "Duplicate compounds remain after the merge.\n"
        f"Rows: {n_rows}\n"
        f"Unique compounds: {n_unique_compounds}"
    )




print("HDAC INHIBITOR COMPOUNDS")
print("=" * 80)

print(
    f"Number of unique HDAC compounds: "
    f"{n_unique_compounds}"
)

print(
    f"Total available profiles: "
    f"{hdac_compounds['n_profiles'].sum():,.0f}"
)

print(
    f"Median profiles per compound: "
    f"{hdac_compounds['n_profiles'].median():.1f}"
)

print(
    f"\nMean multimodal proxy score: "
    f"{hdac_compounds['actual_predictability'].mean():.4f}"
)

print(
    f"Mean structure-predicted score: "
    f"{hdac_compounds['predicted_predictability'].mean():.4f}"
)

print(
    f"Mean signed prediction error: "
    f"{hdac_compounds['prediction_error'].mean():.4f}"
)

print(
    f"Mean absolute structure-model error: "
    f"{hdac_compounds['absolute_error'].mean():.4f}"
)

print(
    f"\nMean raw reconstruction error: "
    f"{hdac_compounds['mean_total_error'].mean():.4f}"
)

print(
    f"Mean source-normalized reconstruction error: "
    f"{hdac_compounds['mean_source_normalized_error_z'].mean():.4f}"
)




display(
    hdac_compounds[
        [
            "compound_embedding_row_index",
            "compound_name",
            "n_profiles",
            "mean_total_error",
            "mean_source_normalized_error_z",
            "source_normalized_inverted_score",
            "actual_predictability",
            "predicted_predictability",
            "prediction_error",
            "absolute_error",
        ]
    ]
    .round(
        {
            "mean_total_error": 4,
            "mean_source_normalized_error_z": 4,
            "source_normalized_inverted_score": 4,
            "actual_predictability": 4,
            "predicted_predictability": 4,
            "prediction_error": 4,
            "absolute_error": 4,
        }
    )
)

HDAC INHIBITOR COMPOUNDS
Number of unique HDAC compounds: 19
Total available profiles: 3,528
Median profiles per compound: 97.0

Mean multimodal proxy score: 0.8800
Mean structure-predicted score: 0.8184
Mean signed prediction error: -0.0616
Mean absolute structure-model error: 0.0817

Mean raw reconstruction error: 0.1124
Mean source-normalized reconstruction error: -0.2880


,compound_embedding_row_index,compound_name,n_profiles,mean_total_error,mean_source_normalized_error_z,source_normalized_inverted_score,actual_predictability,predicted_predictability,prediction_error,absolute_error
0,592,BRD-K68202742; Trichostatin A,1346,0.0569,-1.0932,1.0000,1.0000,0.8271,-0.1729,0.1729
1,1471,Abexinostat; PCI-24781,117,0.0814,-0.7679,0.9791,0.9791,0.8279,-0.1512,0.1512
2,3482,Entinostat; entinostat,410,0.0741,-0.7585,0.9772,0.9772,0.8150,-0.1623,0.1623
3,743,"(3S,6S,9S,12R)-3-[(2S)-butan-2-yl]-6-[(1-metho...",92,0.0805,-0.5506,0.9353,0.9353,0.8464,-0.0889,0.0889
4,3450,Tucidinostat,159,0.0528,-0.5188,0.9289,0.9289,0.8356,-0.0932,0.0932
5,3578,(E)-N-hydroxy-3-[3-(phenylsulfamoyl)phenyl]pro...,400,0.0666,-0.5135,0.9278,0.9278,0.8120,-0.1158,0.1158
6,1473,Resminostat; resminostat,97,0.0993,-0.4742,0.9199,0.9199,0.8370,-0.0829,0.0829
7,946,Pracinostat; SB-939,180,0.1067,-0.4666,0.9184,0.9184,0.8150,-0.1034,0.1034
8,3096,JNJ-26481585; Quisinostat,165,0.0883,-0.4527,0.9156,0.9156,0.8254,-0.0901,0.0901
9,739,Romidepsin; romidepsin,131,0.0987,-0.3879,0.9025,0.9025,0.8400,-0.0625,0.0625


In [9]:
# Summarize what kind of problem, if any, the audit found


print("AUTOMATIC AUDIT INTERPRETATION")
print("=" * 80)


score_formula_passed = (
    max_score_difference < 1e-8
)

aggregation_passed = (
    max_profile_difference == 0
    and max_mean_error_difference < 1e-8
)

handoff_passed = (
    max_handoff_difference < 1e-8
)


if score_formula_passed:
    print(
        "1. PASS: The saved final score matches the "
        "intended inversion formula."
    )
else:
    print(
        "1. FAIL: The final score does not match the "
        "intended inversion formula."
    )


if aggregation_passed:
    print(
        "2. PASS: Signature-level errors were aggregated "
        "to compounds correctly."
    )
else:
    print(
        "2. WARNING: Signature-to-compound aggregation "
        "did not match exactly."
    )


if handoff_passed:
    print(
        "3. PASS: Step 4 used the Step 3 score without "
        "flipping it."
    )
else:
    print(
        "3. FAIL: Step 4's actual target differs from "
        "the Step 3 score."
    )


if hdac_summary.empty:

    print(
        "4. WARNING: No HDAC inhibitor row was found "
        "after the MoA cleaning and n≥5 filter."
    )

else:

    hdac_row = hdac_summary.iloc[0]

    overall_median_proxy = (
        moa_summary["observed_mean"].median()
    )

    hdac_proxy = (
        hdac_row["observed_mean"]
    )

    hdac_observed_rank = int(
        hdac_row[
            "observed_predictability_rank"
        ]
    )

    hdac_error_rank = int(
        hdac_row[
            "prediction_error_rank"
        ]
    )

    print()
    print(
        f"4. HDAC inhibitor multimodal proxy mean: "
        f"{hdac_proxy:.4f}"
    )

    print(
        f"   HDAC proxy rank: "
        f"{hdac_observed_rank} of {len(moa_summary)} "
        f"(1 = highest proxy score)"
    )

    print(
        f"   HDAC structure-model error rank: "
        f"{hdac_error_rank} of {len(moa_summary)} "
        f"(1 = largest error)"
    )


    if hdac_proxy >= overall_median_proxy:

        print(
            "\nINTERPRETATION: HDAC inhibitors have an "
            "above-median multimodal proxy score."
        )

        print(
            "If HDAC also has a large prediction-error rank, "
            "then its appearance on the 'least accurately "
            "predicted' slide reflects failure of the "
            "structure-only regression—not low biological "
            "predictability."
        )

    else:

        print(
            "\nINTERPRETATION: HDAC inhibitors have a "
            "below-median multimodal proxy score."
        )

        print(
            "If the inversion and handoff checks passed, "
            "then the issue is not a simple code flip. "
            "Instead, the reconstruction-based proxy itself "
            "disagrees with the prior HDAC result."
        )


if (
    score_formula_passed
    and aggregation_passed
    and handoff_passed
):
    print(
        "\nOVERALL: No simple inversion or data-handoff bug "
        "was found in the saved pipeline."
    )

    print(
        "The remaining question is whether HDAC inhibitors "
        "have a high proxy score but are poorly predicted by "
        "the structure model, or whether the proxy itself "
        "ranks them unexpectedly."
    )
else:
    print(
        "\nOVERALL: At least one internal pipeline check "
        "failed and should be investigated before interpreting "
        "the biological results."
    )

AUTOMATIC AUDIT INTERPRETATION
1. PASS: The saved final score matches the intended inversion formula.
2. PASS: Signature-level errors were aggregated to compounds correctly.
3. FAIL: Step 4's actual target differs from the Step 3 score.

4. HDAC inhibitor multimodal proxy mean: 0.8800
   HDAC proxy rank: 8 of 52 (1 = highest proxy score)
   HDAC structure-model error rank: 3 of 52 (1 = largest error)

INTERPRETATION: HDAC inhibitors have an above-median multimodal proxy score.
If HDAC also has a large prediction-error rank, then its appearance on the 'least accurately predicted' slide reflects failure of the structure-only regression—not low biological predictability.

OVERALL: At least one internal pipeline check failed and should be investigated before interpreting the biological results.


In [12]:
# Save 

SCORE_AUDIT_PATH = (
    AUDIT_DIR
    / "score_inversion_audit.csv"
)

AGGREGATION_AUDIT_PATH = (
    AUDIT_DIR
    / "signature_to_compound_aggregation_audit.csv"
)

HANDOFF_AUDIT_PATH = (
    AUDIT_DIR
    / "step3_to_step4_target_handoff_audit.csv"
)

MOA_AUDIT_PATH = (
    AUDIT_DIR
    / "moa_predictability_and_prediction_error_audit.csv"
)

HDAC_COMPOUNDS_PATH = (
    AUDIT_DIR
    / "hdac_inhibitor_compound_audit.csv"
)


audit_scores.to_csv(
    SCORE_AUDIT_PATH,
    index=False,
)

aggregation_audit.to_csv(
    AGGREGATION_AUDIT_PATH,
    index=False,
)

step4_handoff.to_csv(
    HANDOFF_AUDIT_PATH,
    index=False,
)

moa_summary.to_csv(
    MOA_AUDIT_PATH,
    index=False,
)

hdac_compounds.to_csv(
    HDAC_COMPOUNDS_PATH,
    index=False,
)


print("Saved all audit outputs to:")
print(AUDIT_DIR)

print("\nFiles:")
for path in [
    SCORE_AUDIT_PATH,
    AGGREGATION_AUDIT_PATH,
    HANDOFF_AUDIT_PATH,
    MOA_AUDIT_PATH,
    HDAC_COMPOUNDS_PATH,
]:
    print(path.name)

Saved all audit outputs to:
/Users/mell/predicting-predictability/data/step3_multimodal/step4_expanded_n20/predictability_direction_audit

Files:
score_inversion_audit.csv
signature_to_compound_aggregation_audit.csv
step3_to_step4_target_handoff_audit.csv
moa_predictability_and_prediction_error_audit.csv
hdac_inhibitor_compound_audit.csv
